# Modeling exploration

Use this notebook to iterate on feature engineering and model behavior locally without Strava API calls.

## Setup

Run once from repo root:
```bash
pip install -r requirements-dev.txt
python -m ipykernel install --user --name train-tomorrow
```

In [ ]:
import tempfile
from pathlib import Path

import pandas as pd

from blurb import generate_blurb, summarize_top_contributors
from features import FEATURE_COLUMNS, prepare_datasets
from model import feature_contributions, predict_tomorrow, train_and_save_models

In [ ]:
def synthetic_activities(days: int = 120) -> pd.DataFrame:
    start = pd.Timestamp.today().normalize() - pd.to_timedelta(days, unit='D')
    rows = []
    for i in range(days):
        day = start + pd.to_timedelta(i, unit='D')
        if i % 2 == 0 or i % 5 == 0:
            rows.append({
                'date': day.date().isoformat(),
                'type': 'Run',
                'moving_time': 1800 + i * 12,
                'distance': 4500 + i * 35,
                'relative_effort': float(20 + (i % 14) * 3),
                'average_watts': float('nan'),
            })
    return pd.DataFrame(rows)

activities = synthetic_activities()
weather = {
    'temp_high': 18.0,
    'temp_low': 11.0,
    'precip_probability': 25.0,
    'wind_speed': 10.0,
}
prepared = prepare_datasets(activities=activities, tomorrow_weather=weather)
prepared.historical[['as_of_date', 'target_date'] + FEATURE_COLUMNS + ['will_train_tomorrow']].tail(5)

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    models = train_and_save_models(prepared.historical, Path(tmpdir))
    prediction = predict_tomorrow(models, prepared.tomorrow_features)
    contributions = feature_contributions(models.classifier, prepared.tomorrow_features)

prediction

In [ ]:
top_contributors = summarize_top_contributors(contributions, top_n=3)
blurb = generate_blurb(
    will_train=prediction['will_train'],
    probability=prediction['probability'],
    predicted_effort=prediction['predicted_effort'],
    top_contributors=top_contributors,
)

pd.DataFrame(top_contributors), blurb